[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/073.%20The%20Vehicle%20Routing%20Problem%20with%20Time%20Windows%20%28VRPTW%29/P73-Tier-10_executed.ipynb)

---


# Problem 73: Vehicle Routing with Time Windows — Strategic Demand-Shaping Simulation

## Tier 10: Proactive Demand and Market Shaping

### Purpose and scope

This notebook builds a segment-based simulation of how pricing, behavioral nudges, and partnership programs can influence customer demand and market metrics over a sequence of months. It creates a synthetic customer population, applies simple impact rules, reports segment and program metrics, and visualizes the simulated trajectory.

The code does **not** formulate or solve a vehicle routing problem with time windows (VRPTW): it contains no vehicles, routes, travel times, delivery time windows, or routing decisions. Its method is a deterministic, rule-based demand-shaping simulation, not an optimization algorithm. The simulation is presented as a strategic market model connected to the notebook's stated logistics context; its results should not be interpreted as a VRPTW solution or as empirical evidence.

### Inputs and outputs

- **Inputs:** fixed market, segment, pricing, nudge, and partnership parameters; synthetic customer profiles generated with the fixed random seed below; and a simulation horizon in months.
- **Outputs:** monthly customer, order, revenue, market-share, and margin metrics; segment and intervention summaries; and a four-panel visualization.
- **Method:** apply pricing, nudge, and partnership impact rules in that order each month, calculate the resulting metrics, update the local customer count, and summarize the trajectory.

### Model assumptions

- The 1,000 generated profiles are a representative sample for segment characteristics, not the entire market. Segment weights are therefore calculated relative to this sample. The market size is a separate modeled population of 10,000 potential customers.
- Price elasticity is represented by a signed coefficient: a negative coefficient means demand tends to move opposite to price. The pricing calculation uses the coefficient multiplied by the segment's proportional price adjustment as an approximate demand response.
- Partnership acquisition rates are proportions of the modeled total market. Acquired customers are capped at the remaining market capacity. Retention is modeled as an 85% baseline with relative improvements, capped at 100%.
- Impact rules are simplified assumptions rather than estimates fitted to observed data. The same pricing, nudge, and partnership effects are applied each month; the model does not estimate learning, competition, capacity, costs, or causal effects.
- The peak-demand calculation reports a monthly demand proportion. Since the modeled intervention effects are constant across months, this model does not create a progressive month-by-month peak shift.

### Simulation equations

For segment \(s\), let \(e_s\) be its mean price elasticity, \(a_s\) its proportional segment-price adjustment, and \(w_s\) its share of the synthetic profile sample. The approximate pricing effects are aggregated as

$$\Delta_D = \sum_s w_s e_s a_s, \qquad
\Delta_V = \sum_s w_s \frac{a_s}{2}, \qquad
\Delta_P = \sum_s w_s (0.1a_s).$$

Here, \(\Delta_D\) is the modeled change in order demand, \(\Delta_V\) is the modeled change in average order value, and \(\Delta_P\) is the modeled change in peak-hour demand. The factor one-half and the factor 0.1 are simplifying assumptions in the simulation, not estimated parameters. In the code, these quantities are accumulated in `_calculate_monthly_metrics` from the segment impacts returned by `_apply_pricing_strategy`.

For nudge \(j\), the modeled effectiveness is its stated effectiveness multiplied by the target segment's mean susceptibility. Its behavior change is that effectiveness multiplied by the incentive value. The code then applies fixed multipliers to estimate frequency, retention, and off-peak effects.

Let \(A_t\) be active customers at the start of month \(t\), \(N_t\) the modeled partnership acquisitions, and \(r_t\) the modeled retention rate. The customer update is

$$A_{t+1} = \min\left(M, \left\lfloor (A_t + N_t)r_t \right\rfloor\right),$$

where \(M\) is total market size. The simulation uses this recurrence month by month. Monthly orders are calculated from active customers, a base order frequency, and the modeled pricing and nudge frequency effects. Monthly revenue is orders multiplied by average order value; monthly profit is revenue multiplied by the modeled profit margin.

This recurrence is a state update: the active-customer count passed from one month becomes the starting count for the next. The simulation has a single forward pass, so for \(T\) months its time complexity is \(O(T)\), apart from processing the fixed profile, nudge, and partnership collections each month. It does not search over decisions or guarantee an optimum.

### How to interpret the results

The printed metrics describe the consequences of the stated synthetic assumptions. They are not forecasts, measured market outcomes, or proof that an intervention caused a change. In particular, a small or zero peak-demand reduction is consistent with the model's constant monthly intervention effects.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
import random
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("default")
np.random.seed(42)
random.seed(42)

## 1. Represent customer and program data

The data classes give each modeled object a clear set of fields. A customer profile stores behavioral characteristics; pricing, nudge, and partnership records store the rules applied by the simulation; and `MarketEcosystem` groups the inputs and market state. The enumerated segment names make segment keys consistent across profile creation, impact calculations, and reporting.

In [ ]:
class CustomerSegment(Enum):
    PRICE_SENSITIVE = "price_sensitive"
    QUALITY_FOCUSED = "quality_focused"
    CONVENIENCE_SEEKER = "convenience_seeker"
    LOYAL_PREMIUM = "loyal_premium"
    OCCASIONAL_USER = "occasional_user"


@dataclass
class CustomerProfile:
    """Behavioral characteristics for one synthetic customer profile."""

    segment: CustomerSegment
    price_elasticity: float
    time_sensitivity: float
    loyalty_score: float
    order_frequency: float
    avg_order_value: float
    nudge_susceptibility: float
    preferred_hours: List[int]


@dataclass
class PricingStrategy:
    """Base prices and proportional pricing adjustments."""

    base_price: float
    peak_premium: float
    off_peak_discount: float
    loyalty_discount: float
    bulk_discount: float
    segment_pricing: Dict[CustomerSegment, float]


@dataclass
class BehavioralNudge:
    """A segment-targeted intervention with a modeled response rate."""

    nudge_type: str
    target_segment: CustomerSegment
    incentive_value: float
    psychological_trigger: str
    effectiveness: float
    duration_days: int


@dataclass
class PartnershipProgram:
    """A partnership with acquisition, retention, and commission assumptions."""

    program_id: str
    partner_type: str
    partner_name: str
    benefits: List[str]
    commission_rate: float
    customer_acquisition_rate: float
    retention_improvement: float


@dataclass
class MarketEcosystem:
    """Customer segments, intervention programs, and modeled market state."""

    customer_segments: Dict[CustomerSegment, List[CustomerProfile]]
    pricing_strategy: PricingStrategy
    behavioral_nudges: List[BehavioralNudge]
    partnership_programs: List[PartnershipProgram]
    market_metrics: Dict
    competitive_position: float
    market_share: float
    profit_margin: float

## 2. Build and simulate the market ecosystem

`DemandShapingOptimizer` is retained as the class name used by the example, but its role here is a rule-based simulation engine: it does not optimize a mathematical objective. Its construction methods create the synthetic inputs. The simulation then applies pricing, nudges, and partnerships, calculates monthly metrics, and records the market trajectory. Separating these steps makes it possible to inspect the assumptions and intermediate impact calculations.

In [ ]:
class DemandShapingOptimizer:
    """Rule-based simulator for the stated demand-shaping assumptions."""

    def __init__(self):
        self.market_size = 10000
        self.current_penetration = 0.15
        self.target_penetration = 0.35

    def create_market_ecosystem(self) -> MarketEcosystem:
        """Create customer profiles and the intervention programs."""
        customer_segments = self._create_customer_segments()
        pricing_strategy = self._create_pricing_strategy()
        behavioral_nudges = self._create_behavioral_nudges()
        partnership_programs = self._create_partnership_programs()

        market_metrics = {
            "total_customers": self.market_size,
            "active_customers": int(self.market_size * self.current_penetration),
            "monthly_orders": 0,
            "avg_order_value": 0.0,
            "peak_hour_demand": 0.4,
            "off_peak_demand": 0.6,
            "customer_acquisition_cost": 15.0,
            "customer_lifetime_value": 450.0,
        }

        return MarketEcosystem(
            customer_segments=customer_segments,
            pricing_strategy=pricing_strategy,
            behavioral_nudges=behavioral_nudges,
            partnership_programs=partnership_programs,
            market_metrics=market_metrics,
            competitive_position=0.25,
            market_share=self.current_penetration,
            profit_margin=0.12,
        )

    def _create_customer_segments(self) -> Dict[CustomerSegment, List[CustomerProfile]]:
        """Generate the fixed-size synthetic profile sample by segment."""
        segment_specs = {
            CustomerSegment.PRICE_SENSITIVE: {
                "count": 350, "elasticity": (-0.15, 0.05),
                "time": (0.2, 0.6), "loyalty": (0.1, 0.4),
                "frequency": (2, 6), "value": (15, 35),
                "susceptibility": (0.6, 0.9),
                "hours": [11, 12, 13, 14, 18, 19],
            },
            CustomerSegment.QUALITY_FOCUSED: {
                "count": 250, "elasticity": (-0.05, 0.03),
                "time": (0.3, 0.7), "loyalty": (0.4, 0.7),
                "frequency": (4, 8), "value": (35, 60),
                "susceptibility": (0.3, 0.6),
                "hours": [12, 13, 19, 20, 21],
            },
            CustomerSegment.CONVENIENCE_SEEKER: {
                "count": 200, "elasticity": (-0.08, 0.04),
                "time": (0.6, 0.9), "loyalty": (0.3, 0.6),
                "frequency": (6, 12), "value": (25, 45),
                "susceptibility": (0.5, 0.8),
                "hours": [11, 12, 18, 19, 20],
            },
            CustomerSegment.LOYAL_PREMIUM: {
                "count": 150, "elasticity": (-0.03, 0.02),
                "time": (0.4, 0.8), "loyalty": (0.7, 0.95),
                "frequency": (8, 15), "value": (45, 80),
                "susceptibility": (0.2, 0.5),
                "hours": [12, 13, 19, 20, 21],
            },
            CustomerSegment.OCCASIONAL_USER: {
                "count": 50, "elasticity": (-0.12, 0.06),
                "time": (0.1, 0.5), "loyalty": (0.05, 0.25),
                "frequency": (1, 3), "value": (20, 40),
                "susceptibility": (0.7, 1.0),
                "hours": [11, 12, 18, 19, 20],
            },
        }

        segments = {}
        for segment, spec in segment_specs.items():
            profiles = []
            for _ in range(spec["count"]):
                profiles.append(
                    CustomerProfile(
                        segment=segment,
                        price_elasticity=np.random.normal(*spec["elasticity"]),
                        time_sensitivity=np.random.uniform(*spec["time"]),
                        loyalty_score=np.random.uniform(*spec["loyalty"]),
                        order_frequency=np.random.uniform(*spec["frequency"]),
                        avg_order_value=np.random.uniform(*spec["value"]),
                        nudge_susceptibility=np.random.uniform(*spec["susceptibility"]),
                        preferred_hours=spec["hours"],
                    )
                )
            segments[segment] = profiles
        return segments

    def _create_pricing_strategy(self) -> PricingStrategy:
        """Set the base price and segment-level proportional adjustments."""
        segment_pricing = {
            CustomerSegment.PRICE_SENSITIVE: -0.15,
            CustomerSegment.QUALITY_FOCUSED: 0.05,
            CustomerSegment.CONVENIENCE_SEEKER: 0.02,
            CustomerSegment.LOYAL_PREMIUM: -0.10,
            CustomerSegment.OCCASIONAL_USER: -0.08,
        }
        return PricingStrategy(
            base_price=25.0,
            peak_premium=0.20,
            off_peak_discount=0.15,
            loyalty_discount=0.10,
            bulk_discount=0.05,
            segment_pricing=segment_pricing,
        )

    def _create_behavioral_nudges(self) -> List[BehavioralNudge]:
        """Create the segment-targeted interventions used in the simulation."""
        return [
            BehavioralNudge("off_peak_gamification", CustomerSegment.PRICE_SENSITIVE,
                            0.15, "achievement_and_rewards", 0.65, 90),
            BehavioralNudge("loyalty_points", CustomerSegment.LOYAL_PREMIUM,
                            0.10, "status_and_exclusivity", 0.75, 180),
            BehavioralNudge("convenience_bundles", CustomerSegment.CONVENIENCE_SEEKER,
                            0.12, "simplicity_and_time_saving", 0.58, 60),
            BehavioralNudge("quality_assurance", CustomerSegment.QUALITY_FOCUSED,
                            0.08, "trust_and_reliability", 0.52, 120),
            BehavioralNudge("new_user_welcome", CustomerSegment.OCCASIONAL_USER,
                            0.20, "fear_of_missing_out", 0.82, 30),
        ]

    def _create_partnership_programs(self) -> List[PartnershipProgram]:
        """Create partnership assumptions; acquisition rates are proportions."""
        return [
            PartnershipProgram(
                "REST_001", "restaurant", "Local Eats Partnership",
                ["increased_order_volume", "marketing_exposure", "data_insights"],
                0.08, 0.15, 0.12,
            ),
            PartnershipProgram(
                "CORP_001", "corporate", "Corporate Lunch Program",
                ["stable_revenue", "bulk_orders", "brand_loyalty"],
                0.12, 0.25, 0.20,
            ),
            PartnershipProgram(
                "LOY_001", "loyalty", "Premium Rewards Network",
                ["cross_promotion", "premium_customers", "data_sharing"],
                0.05, 0.18, 0.25,
            ),
        ]

    def simulate_demand_shaping(
        self, ecosystem: MarketEcosystem, months: int = 12
    ) -> Dict:
        """Apply the impact rules month by month and return their metrics."""
        if months <= 0:
            raise ValueError("months must be a positive integer")

        results = {
            "monthly_metrics": [],
            "segment_performance": {},
            "nudge_effectiveness": {},
            "partnership_impact": {},
            "market_evolution": {},
        }
        active_customers = ecosystem.market_metrics["active_customers"]

        for month in range(months):
            pricing_impact = self._apply_pricing_strategy(ecosystem)
            nudge_impact = self._apply_behavioral_nudges(ecosystem, month)
            partnership_impact = self._apply_partnership_programs(ecosystem)
            monthly_metrics = self._calculate_monthly_metrics(
                ecosystem,
                active_customers,
                pricing_impact,
                nudge_impact,
                partnership_impact,
            )
            active_customers = monthly_metrics["active_customers"]
            results["monthly_metrics"].append(monthly_metrics)

        results["segment_performance"] = self._analyze_segment_performance(ecosystem)
        results["nudge_effectiveness"] = self._analyze_nudge_effectiveness(ecosystem)
        results["partnership_impact"] = self._analyze_partnership_impact(ecosystem)
        results["market_evolution"] = self._analyze_market_evolution(
            results["monthly_metrics"]
        )

        final_metrics = results["monthly_metrics"][-1]
        ecosystem.market_metrics.update(final_metrics)
        ecosystem.market_share = final_metrics["market_share"]
        ecosystem.profit_margin = final_metrics["profit_margin"]
        ecosystem.competitive_position = min(0.8, final_metrics["market_share"] * 2)
        return results

    def _apply_pricing_strategy(self, ecosystem: MarketEcosystem) -> Dict:
        """Estimate segment demand, peak, and order-value effects."""
        pricing = ecosystem.pricing_strategy
        segment_impact = {}
        for segment, profiles in ecosystem.customer_segments.items():
            if not profiles:
                continue
            avg_elasticity = np.mean([p.price_elasticity for p in profiles])
            adjustment = pricing.segment_pricing[segment]
            demand_change = avg_elasticity * adjustment
            segment_impact[segment] = {
                "demand_change": demand_change,
                "peak_shift": 0.1 * adjustment,
                "revenue_impact": adjustment * (1 + demand_change),
                "avg_order_change": adjustment * 0.5,
            }
        return segment_impact

    def _apply_behavioral_nudges(
        self, ecosystem: MarketEcosystem, month: int
    ) -> Dict:
        """Estimate the active nudge effects using segment susceptibility."""
        nudge_impact = {}
        for nudge in ecosystem.behavioral_nudges:
            if month >= nudge.duration_days / 30:
                continue
            profiles = ecosystem.customer_segments.get(nudge.target_segment, [])
            if not profiles:
                continue
            susceptibility = np.mean([p.nudge_susceptibility for p in profiles])
            effectiveness = nudge.effectiveness * susceptibility
            behavior_change = effectiveness * nudge.incentive_value
            nudge_impact[nudge.nudge_type] = {
                "effectiveness": effectiveness,
                "behavior_change": behavior_change,
                "order_frequency_change": behavior_change * 0.3,
                "retention_improvement": behavior_change * 0.2,
                "off_peak_shift": (
                    behavior_change * 0.4
                    if "off_peak" in nudge.nudge_type
                    else 0.0
                ),
            }
        return nudge_impact

    def _apply_partnership_programs(self, ecosystem: MarketEcosystem) -> Dict:
        """Convert partnership assumptions into modeled monthly effects."""
        impacts = {}
        total_market = ecosystem.market_metrics["total_customers"]
        for program in ecosystem.partnership_programs:
            impacts[program.program_id] = {
                "new_customers": (
                    total_market * program.customer_acquisition_rate
                ),
                "retention_boost": program.retention_improvement,
                "revenue_impact": 1 - program.commission_rate,
                "partner_type": program.partner_type,
            }
        return impacts

    def _calculate_monthly_metrics(
        self,
        ecosystem: MarketEcosystem,
        active_customers: int,
        pricing_impact: Dict,
        nudge_impact: Dict,
        partnership_impact: Dict,
    ) -> Dict:
        """Aggregate intervention effects and calculate one month's metrics."""
        base_orders_per_customer = 8.5
        base_avg_order_value = 32.0
        base_peak_demand = 0.4

        profile_count = sum(
            len(profiles) for profiles in ecosystem.customer_segments.values()
        )
        pricing_orders_change = 0.0
        pricing_value_change = 0.0
        pricing_peak_shift = 0.0
        for segment, impact in pricing_impact.items():
            segment_weight = len(ecosystem.customer_segments[segment]) / profile_count
            pricing_orders_change += impact["demand_change"] * segment_weight
            pricing_value_change += impact["avg_order_change"] * segment_weight
            pricing_peak_shift += impact["peak_shift"] * segment_weight

        nudge_frequency_change = sum(
            impact["order_frequency_change"] * 0.2
            for impact in nudge_impact.values()
        )
        nudge_retention_boost = sum(
            impact["retention_improvement"] * 0.15
            for impact in nudge_impact.values()
        )
        nudge_off_peak_shift = sum(
            impact["off_peak_shift"] * 0.3
            for impact in nudge_impact.values()
        )

        partnership_customers = sum(
            impact["new_customers"] for impact in partnership_impact.values()
        )
        partnership_retention = (
            np.mean([
                impact["retention_boost"]
                for impact in partnership_impact.values()
            ])
            if partnership_impact
            else 0.0
        )
        partnership_revenue = (
            np.mean([
                impact["revenue_impact"]
                for impact in partnership_impact.values()
            ])
            if partnership_impact
            else 1.0
        )

        monthly_orders = int(
            active_customers
            * base_orders_per_customer
            * (1 + pricing_orders_change + nudge_frequency_change)
        )
        avg_order_value = (
            base_avg_order_value * (1 + pricing_value_change) * partnership_revenue
        )
        peak_hour_demand = float(
            np.clip(
                base_peak_demand + pricing_peak_shift + nudge_off_peak_shift,
                0.0,
                1.0,
            )
        )

        market_size = ecosystem.market_metrics["total_customers"]
        new_active_customers = min(
            market_size, active_customers + int(partnership_customers)
        )
        retention_rate = min(
            1.0,
            0.85 * (1 + nudge_retention_boost + partnership_retention),
        )
        final_active_customers = min(
            market_size, int(new_active_customers * retention_rate)
        )
        market_share = final_active_customers / market_size
        profit_margin = 0.12 + (market_share - 0.15) * 0.3
        monthly_revenue = monthly_orders * avg_order_value

        return {
            "active_customers": final_active_customers,
            "monthly_orders": monthly_orders,
            "avg_order_value": avg_order_value,
            "peak_hour_demand": peak_hour_demand,
            "off_peak_demand": 1 - peak_hour_demand,
            "market_share": market_share,
            "profit_margin": profit_margin,
            "monthly_revenue": monthly_revenue,
            "monthly_profit": monthly_revenue * profit_margin,
        }

    def _analyze_segment_performance(self, ecosystem: MarketEcosystem) -> Dict:
        """Summarize the synthetic profile characteristics of each segment."""
        segment_analysis = {}
        peak_hours = [11, 12, 18, 19, 20]
        for segment, profiles in ecosystem.customer_segments.items():
            if not profiles:
                continue
            preferred_fraction = sum(
                hour in profile.preferred_hours
                for profile in profiles
                for hour in peak_hours
            ) / (len(profiles) * len(peak_hours))
            segment_analysis[segment.value] = {
                "segment_size": len(profiles),
                "avg_elasticity": float(np.mean(
                    [p.price_elasticity for p in profiles]
                )),
                "avg_loyalty": float(np.mean(
                    [p.loyalty_score for p in profiles]
                )),
                "avg_frequency": float(np.mean(
                    [p.order_frequency for p in profiles]
                )),
                "avg_value": float(np.mean(
                    [p.avg_order_value for p in profiles]
                )),
                "peak_preference": preferred_fraction,
            }
        return segment_analysis

    def _analyze_nudge_effectiveness(self, ecosystem: MarketEcosystem) -> Dict:
        """Report assumed nudge effectiveness adjusted for susceptibility."""
        analysis = {}
        for nudge in ecosystem.behavioral_nudges:
            profiles = ecosystem.customer_segments.get(nudge.target_segment, [])
            if not profiles:
                continue
            susceptibility = float(np.mean(
                [p.nudge_susceptibility for p in profiles]
            ))
            analysis[nudge.nudge_type] = {
                "target_segment": nudge.target_segment.value,
                "base_effectiveness": nudge.effectiveness,
                "actual_effectiveness": nudge.effectiveness * susceptibility,
                "susceptibility_factor": susceptibility,
                "incentive_value": nudge.incentive_value,
                "psychological_trigger": nudge.psychological_trigger,
            }
        return analysis

    def _analyze_partnership_impact(self, ecosystem: MarketEcosystem) -> Dict:
        """Report the assumed acquisition, retention, and commission terms."""
        return {
            program.program_id: {
                "partner_name": program.partner_name,
                "partner_type": program.partner_type,
                "commission_rate": program.commission_rate,
                "acquisition_rate": program.customer_acquisition_rate,
                "retention_improvement": program.retention_improvement,
                "benefits": program.benefits,
            }
            for program in ecosystem.partnership_programs
        }

    def _analyze_market_evolution(self, monthly_metrics: List[Dict]) -> Dict:
        """Compare first- and last-month metrics in the simulated trajectory."""
        if not monthly_metrics:
            return {}
        first = monthly_metrics[0]
        last = monthly_metrics[-1]

        def percentage_change(initial: float, final: float) -> float:
            if initial == 0:
                return 0.0 if final == 0 else float("inf")
            return (final - initial) / initial * 100

        peak_reduction = percentage_change(
            first["peak_hour_demand"], last["peak_hour_demand"]
        )
        peak_reduction = -peak_reduction
        return {
            "customer_growth_rate": percentage_change(
                first["active_customers"], last["active_customers"]
            ),
            "revenue_growth_rate": percentage_change(
                first["monthly_revenue"], last["monthly_revenue"]
            ),
            "margin_improvement": (
                last["profit_margin"] - first["profit_margin"]
            ) * 100,
            "peak_demand_reduction": peak_reduction,
            "initial_customers": first["active_customers"],
            "final_customers": last["active_customers"],
            "initial_revenue": first["monthly_revenue"],
            "final_revenue": last["monthly_revenue"],
            "months_analyzed": len(monthly_metrics),
        }

## 3. Create the ecosystem and inspect its assumptions

The profile counts sum to the synthetic sample size. The displayed segment averages describe that sample; they are not observed statistics from the full potential market. Reviewing these values before simulation helps connect the input assumptions to the impact rules that follow.

In [ ]:
optimizer = DemandShapingOptimizer()
market_ecosystem = optimizer.create_market_ecosystem()

print("Market ecosystem assumptions")
print(f"  Total potential market: {optimizer.market_size:,} customers")
print(f"  Initial penetration: {optimizer.current_penetration:.1%}")
print(f"  Target penetration assumption: {optimizer.target_penetration:.1%}")
print(f"  Synthetic customer segments: {len(market_ecosystem.customer_segments)}")
print(f"  Behavioral nudges: {len(market_ecosystem.behavioral_nudges)}")
print(f"  Partnership programs: {len(market_ecosystem.partnership_programs)}")

print("\nSegment profile summaries")
for segment, profiles in market_ecosystem.customer_segments.items():
    print(
        f"  {segment.value}: {len(profiles)} profiles; "
        f"mean elasticity={np.mean([p.price_elasticity for p in profiles]):.3f}; "
        f"mean loyalty={np.mean([p.loyalty_score for p in profiles]):.3f}; "
        f"mean orders/month={np.mean([p.order_frequency for p in profiles]):.1f}; "
        f"mean order value=${np.mean([p.avg_order_value for p in profiles]):.2f}"
    )

pricing = market_ecosystem.pricing_strategy
print("\nPricing assumptions")
print(f"  Base price: ${pricing.base_price:.2f}")
print(f"  Peak premium: {pricing.peak_premium:.1%}")
print(f"  Off-peak discount: {pricing.off_peak_discount:.1%}")
print(f"  Loyalty discount: {pricing.loyalty_discount:.1%}")
print(f"  Bulk discount: {pricing.bulk_discount:.1%}")
print("  Segment adjustments:")
for segment, adjustment in pricing.segment_pricing.items():
    label = "discount" if adjustment < 0 else "premium"
    print(f"    {segment.value}: {adjustment:+.1%} ({label})")

## 4. Run the monthly simulation

Each iteration uses the active-customer count produced by the preceding month. Pricing, nudge, and partnership impacts are calculated before the monthly metrics, following the stated application order. The target penetration is reported as an assumption; it is not enforced as a constraint or optimized toward.

In [ ]:
simulation_results = optimizer.simulate_demand_shaping(
    market_ecosystem,
    months=12,
)
evolution = simulation_results["market_evolution"]
final_metrics = simulation_results["monthly_metrics"][-1]

market_advantage = (
    (final_metrics["market_share"] - optimizer.current_penetration)
    / optimizer.current_penetration
    * 100
)
profit_advantage = (
    (final_metrics["profit_margin"] - 0.12) / 0.12 * 100
)

print("Demand-shaping simulation summary")
print(f"  Months simulated: {evolution['months_analyzed']}")
print(f"  First-month active customers: {evolution['initial_customers']:,}")
print(f"  Last-month active customers: {evolution['final_customers']:,}")
print(f"  First-to-last customer change: {evolution['customer_growth_rate']:.1f}%")
print(f"  First-to-last revenue change: {evolution['revenue_growth_rate']:.1f}%")
print(f"  Profit-margin change: {evolution['margin_improvement']:.2f} percentage points")
print(f"  Peak-demand reduction: {evolution['peak_demand_reduction']:.2f}%")
print(f"  Final market share: {final_metrics['market_share']:.1%}")
print(f"  Final modeled profit margin: {final_metrics['profit_margin']:.1%}")
print(f"  Final modeled monthly profit: ${final_metrics['monthly_profit']:,.2f}")
print(f"  Market-share change vs. initial penetration: {market_advantage:.1f}%")
print(f"  Margin change vs. 12% baseline: {profit_advantage:.1f}%")

## 5. Compare customer segments

The table summarizes the synthetic segment profiles, not segment-level outcomes from the monthly simulation. A more negative mean elasticity indicates a stronger modeled demand response to a price change. Peak preference is the share of the five listed reference hours that appear in a segment's preferred-hour lists.

In [ ]:
segment_performance = simulation_results["segment_performance"]
segment_data = []
for segment_name, metrics in segment_performance.items():
    segment_data.append(
        {
            "Segment": segment_name.replace("_", " ").title(),
            "Profiles": metrics["segment_size"],
            "Mean Price Elasticity": f"{metrics['avg_elasticity']:.3f}",
            "Mean Loyalty": f"{metrics['avg_loyalty']:.3f}",
            "Mean Orders per Month": f"{metrics['avg_frequency']:.1f}",
            "Mean Order Value": f"${metrics['avg_value']:.2f}",
            "Peak-Hour Preference": f"{metrics['peak_preference']:.1%}",
        }
    )

segment_df = pd.DataFrame(segment_data)
print(segment_df.to_string(index=False))

best_loyalty = max(
    segment_performance.items(), key=lambda item: item[1]["avg_loyalty"]
)
most_price_sensitive = min(
    segment_performance.items(), key=lambda item: item[1]["avg_elasticity"]
)
highest_value = max(
    segment_performance.items(), key=lambda item: item[1]["avg_value"]
)
print("\nProfile-based segment observations")
print(f"  Highest mean loyalty: {best_loyalty[0]} ({best_loyalty[1]['avg_loyalty']:.3f})")
print(
    f"  Most price-sensitive mean profile: {most_price_sensitive[0]} "
    f"({most_price_sensitive[1]['avg_elasticity']:.3f})"
)
print(f"  Highest mean order value: {highest_value[0]} (${highest_value[1]['avg_value']:.2f})")

print("\nRule-based segment suggestions")
for segment, metrics in segment_performance.items():
    if metrics["avg_loyalty"] > 0.7:
        suggestion = "consider retention programs and premium services"
    elif metrics["avg_elasticity"] < -0.1:
        suggestion = "consider testing targeted prices and promotions"
    elif metrics["avg_frequency"] > 10:
        suggestion = "consider evaluating subscription or membership options"
    elif metrics["peak_preference"] > 0.7:
        suggestion = "consider reviewing peak-hour capacity needs"
    else:
        suggestion = "consider testing acquisition and conversion strategies"
    print(f"  {segment.replace('_', ' ').title()}: {suggestion}")

## 6. Inspect behavioral nudge assumptions

Adjusted effectiveness is calculated as base effectiveness multiplied by the target segment's mean susceptibility. This is a modeled index between zero and one under the stated inputs; it is not a measured response rate. Since susceptibility is at most one, adjusted effectiveness cannot exceed the corresponding base value.

In [ ]:
nudge_effectiveness = simulation_results["nudge_effectiveness"]
nudge_data = []
for nudge_type, metrics in nudge_effectiveness.items():
    nudge_data.append(
        {
            "Nudge": nudge_type.replace("_", " ").title(),
            "Target Segment": metrics["target_segment"].replace("_", " ").title(),
            "Base Effectiveness": f"{metrics['base_effectiveness']:.1%}",
            "Adjusted Effectiveness": f"{metrics['actual_effectiveness']:.1%}",
            "Mean Susceptibility": f"{metrics['susceptibility_factor']:.3f}",
            "Incentive": f"{metrics['incentive_value']:.1%}",
            "Trigger": metrics["psychological_trigger"].replace("_", " ").title(),
        }
    )
print(pd.DataFrame(nudge_data).to_string(index=False))

effectiveness_gaps = [
    (
        nudge_type,
        metrics["actual_effectiveness"] - metrics["base_effectiveness"],
    )
    for nudge_type, metrics in nudge_effectiveness.items()
]
best_adjusted = max(effectiveness_gaps, key=lambda item: item[1])
largest_gap = min(effectiveness_gaps, key=lambda item: item[1])
print("\nAdjusted-effectiveness comparisons")
print(f"  Smallest reduction from base: {best_adjusted[0]} ({best_adjusted[1]:.1%})")
print(f"  Largest reduction from base: {largest_gap[0]} ({largest_gap[1]:.1%})")

trigger_analysis = defaultdict(list)
for metrics in nudge_effectiveness.values():
    trigger_analysis[metrics["psychological_trigger"]].append(
        metrics["actual_effectiveness"]
    )
print("\nMean adjusted effectiveness by trigger")
for trigger, values in trigger_analysis.items():
    print(f"  {trigger.replace('_', ' ').title()}: {np.mean(values):.1%}")

## 7. Inspect partnership assumptions

The table reports the input terms assigned to each partnership. The acquisition ROI calculation below is deliberately simplified: it compares acquisition rate multiplied by the assumed customer lifetime value with the assumed acquisition cost. It omits timing, customer-level acquisition counts, commissions, operating costs, and uncertainty, so it is an illustrative index rather than a full financial return calculation.

In [ ]:
partnership_impact = simulation_results["partnership_impact"]
partnership_data = []
for program_id, metrics in partnership_impact.items():
    partnership_data.append(
        {
            "Program": program_id,
            "Partner": metrics["partner_name"],
            "Type": metrics["partner_type"].title(),
            "Commission": f"{metrics['commission_rate']:.1%}",
            "Acquisition Rate": f"{metrics['acquisition_rate']:.1%}",
            "Retention Improvement": f"{metrics['retention_improvement']:.1%}",
            "Illustrative Benefits": ", ".join(
                benefit.replace("_", " ").title()
                for benefit in metrics["benefits"][:2]
            ),
        }
    )
print(pd.DataFrame(partnership_data).to_string(index=False))

print("\nSimplified acquisition comparisons")
for program_id, metrics in partnership_impact.items():
    acquisition_cost = market_ecosystem.market_metrics["customer_acquisition_cost"]
    lifetime_value = market_ecosystem.market_metrics["customer_lifetime_value"]
    modeled_value = metrics["acquisition_rate"] * lifetime_value
    acquisition_roi = (modeled_value - acquisition_cost) / acquisition_cost * 100
    revenue_retention = (1 - metrics["commission_rate"]) * 100
    print(f"  {program_id}:")
    print(f"    Simplified acquisition ROI index: {acquisition_roi:.1f}%")
    print(f"    Revenue retained after commission: {revenue_retention:.1f}%")
    print(f"    Assumed retention improvement: {metrics['retention_improvement']:.1%}")

type_performance = defaultdict(list)
for metrics in partnership_impact.values():
    type_performance[metrics["partner_type"]].append(metrics["acquisition_rate"])
print("\nMean assumed acquisition rate by partner type")
for partner_type, rates in type_performance.items():
    print(f"  {partner_type.title()}: {np.mean(rates):.1%}")

## 8. Visualize the simulated metrics

The panels show monthly modeled metrics, profile counts by segment, stated versus susceptibility-adjusted nudge effectiveness, and the peak/off-peak demand proportions. Separate axes are used where units differ. Because the intervention effects are held constant over time, a flat peak-demand series is an expected consequence of these assumptions, not evidence of an evolving demand response.

In [ ]:
def visualize_demand_shaping_results(
    results: Dict, ecosystem: MarketEcosystem
) -> None:
    """Plot the monthly trajectory and the modeled segment and nudge metrics."""
    monthly_metrics = results["monthly_metrics"]
    if not monthly_metrics:
        raise ValueError("The results contain no monthly metrics to plot")

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    ax1, ax2, ax3, ax4 = axes.flat
    months = np.arange(1, len(monthly_metrics) + 1)

    customers = [metric["active_customers"] for metric in monthly_metrics]
    revenue_thousands = [metric["monthly_revenue"] / 1000 for metric in monthly_metrics]
    margins = [metric["profit_margin"] * 100 for metric in monthly_metrics]
    ax1_twin = ax1.twinx()
    customer_line = ax1.plot(
        months, customers, "b-o", label="Active customers"
    )
    revenue_line = ax1.plot(
        months, revenue_thousands, "g-s", label="Revenue ($ thousands)"
    )
    margin_line = ax1_twin.plot(
        months, margins, "r-^", label="Profit margin (%)"
    )
    ax1.set_xlabel("Month")
    ax1.set_ylabel("Customers; revenue ($ thousands)")
    ax1_twin.set_ylabel("Profit margin (%)")
    ax1.set_title("Monthly Market Metrics")
    ax1.grid(True, alpha=0.3)
    lines = customer_line + revenue_line + margin_line
    ax1.legend(lines, [line.get_label() for line in lines], loc="best")

    segment_performance = results["segment_performance"]
    segment_names = list(segment_performance)
    segment_sizes = [
        segment_performance[name]["segment_size"] for name in segment_names
    ]
    segment_loyalty = [
        segment_performance[name]["avg_loyalty"] for name in segment_names
    ]
    bars = ax2.bar(segment_names, segment_sizes, color=plt.cm.Set3(
        np.linspace(0, 1, len(segment_names))
    ))
    ax2.set_xlabel("Customer segment")
    ax2.set_ylabel("Synthetic profile count")
    ax2.set_title("Synthetic Segment Sample")
    ax2.tick_params(axis="x", labelrotation=30)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, loyalty in zip(bars, segment_loyalty):
        ax2.annotate(
            f"Mean loyalty: {loyalty:.2f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    nudge_effectiveness = results["nudge_effectiveness"]
    nudge_names = list(nudge_effectiveness)
    positions = np.arange(len(nudge_names))
    width = 0.36
    base_values = [
        nudge_effectiveness[name]["base_effectiveness"] for name in nudge_names
    ]
    adjusted_values = [
        nudge_effectiveness[name]["actual_effectiveness"]
        for name in nudge_names
    ]
    ax3.bar(
        positions - width / 2, base_values, width,
        label="Base effectiveness", color="steelblue"
    )
    ax3.bar(
        positions + width / 2, adjusted_values, width,
        label="Adjusted effectiveness", color="seagreen"
    )
    ax3.set_xlabel("Behavioral nudge")
    ax3.set_ylabel("Effectiveness index")
    ax3.set_title("Nudge Effectiveness Assumptions")
    ax3.set_xticks(positions)
    ax3.set_xticklabels(
        [name.replace("_", " ").title() for name in nudge_names],
        rotation=35,
        ha="right",
    )
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, axis="y", alpha=0.3)

    peak_demand = [metric["peak_hour_demand"] for metric in monthly_metrics]
    off_peak_demand = [metric["off_peak_demand"] for metric in monthly_metrics]
    ax4.plot(months, peak_demand, "r-o", label="Peak-hour demand")
    ax4.plot(months, off_peak_demand, "g-s", label="Off-peak demand")
    ax4.set_xlabel("Month")
    ax4.set_ylabel("Share of modeled demand")
    ax4.set_title("Peak and Off-Peak Demand Shares")
    ax4.set_ylim(0, 1)
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_demand_shaping_results(simulation_results, market_ecosystem)

# Summary and conclusions

## What the simulation demonstrates

The notebook represents five customer segments with synthetic behavioral profiles and applies simple rules for segment pricing, time-limited nudges, and partnership programs. A monthly recurrence updates the active-customer count, while formulas convert the modeled intervention effects into orders, average order value, revenue, market share, and profit margin. Tables and plots expose both the assumptions and the resulting simulated metrics.

Interpret results as consequences of the chosen rules. For example, segment elasticity influences the modeled pricing response; nudge susceptibility scales the stated effectiveness; and partnership acquisition rates contribute to customer additions, subject to the market-size cap. The reported profit margin is a rule based on market share, not a cost-accounting calculation. The simplified partnership ROI index is not a complete financial analysis.

## What the simulation does not establish

The profiles and impact coefficients are synthetic assumptions, not observations or calibrated estimates. The simulation does not estimate uncertainty, infer causality, or account for market competition, service capacity, operational costs, customer-level behavior, or time-varying effects. Its constant monthly intervention effects cannot establish a progressive shift in demand. Recommendations printed from threshold rules are prompts for investigation, not validated business decisions.

This notebook does not solve the VRPTW. A VRPTW model would require delivery locations, travel times, vehicle and route decisions, and customer time-window constraints. Those elements are outside the implemented demand-shaping simulation and are not inferred from its market metrics.

## Further work within this modeling scope

The simulation could be extended with additional measured inputs, explicit uncertainty ranges, capacity limits, and time-varying response assumptions. Such extensions would require new data and clearly stated modeling choices; the current outputs alone do not justify specific values or performance claims.